In [1]:
import warnings
warnings.filterwarnings('ignore')

import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from glob import glob
from tqdm import tqdm

import tensorflow as tf
from tensorflow.keras.applications import DenseNet201, ResNet50, InceptionV3
from tensorflow.keras.applications.resnet50 import preprocess_input as preprocess_resnet
from tensorflow.keras.applications.inception_v3 import preprocess_input as preprocess_inception
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing import image

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, roc_auc_score
from imblearn.over_sampling import SMOTE
import xgboost as xgb
from sklearn.svm import SVC
import joblib
from PIL import Image, ImageFilter

In [2]:

DATA_DIR = "./natural_disaster_dataset"
TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR = os.path.join(DATA_DIR, "val")
TEST_DIR = os.path.join(DATA_DIR, "test")

IMG_SIZE = (224,224)
BATCH_SIZE = 16
EPOCHS_STAGE1 = 10
EPOCHS_STAGE2 = 10
LR = 1e-4
RANDOM_STATE = 42
TOPK = 300
os.makedirs("DRI_outputs", exist_ok=True)


In [3]:
def collect(root):
    items=[]
    for cls in sorted(os.listdir(root)):
        p = os.path.join(root, cls)
        if os.path.isdir(p):
            for ext in ('*.jpg','*.jpeg','*.png','*.bmp'):
                items += [(f,cls) for f in glob(os.path.join(p,ext))]
    return items

if not os.path.isdir(TRAIN_DIR):
    raise FileNotFoundError(f"{TRAIN_DIR} missing.")

train_items = collect(TRAIN_DIR)
val_items = collect(VAL_DIR)
test_items = collect(TEST_DIR)

print("Train:",len(train_items),"Val:",len(val_items),"Test:",len(test_items))
CLASSES = sorted(list({l for _,l in train_items}))
print("Classes:", CLASSES)


Train: 3322 Val: 444 Test: 662
Classes: ['cyclone', 'earthquake', 'flood', 'wildfire']


In [4]:
from PIL import Image, ImageFilter
def denoise_image(img_pil):
    return img_pil.filter(ImageFilter.MedianFilter(size=3))
def load_img_array(path, target=IMG_SIZE, denoise=True):
    img = Image.open(path).convert('RGB')
    if denoise: img = denoise_image(img)
    img = img.resize(target)
    arr = np.array(img).astype(np.float32)
    return arr
def preprocess_for_backbone(arr, backbone):
    if backbone=='densenet':
        # DenseNet uses ImageNet scaling like resnet
        return preprocess_resnet(arr.copy())
    if backbone=='resnet':
        return preprocess_resnet(arr.copy())
    if backbone=='inception':
        return preprocess_inception(arr.copy())
    return arr/255.0


In [5]:

from tensorflow.keras.applications import DenseNet201
def get_densenet():
    base = DenseNet201(weights='imagenet', include_top=False, input_shape=(IMG_SIZE[0],IMG_SIZE[1],3))
    out = GlobalAveragePooling2D()(base.output)
    m = Model(inputs=base.input, outputs=out)
    for layer in m.layers: layer.trainable=False
    return m

def get_resnet(): 
    base = ResNet50(weights='imagenet', include_top=False, input_shape=(IMG_SIZE[0],IMG_SIZE[1],3))
    out = GlobalAveragePooling2D()(base.output); m=Model(base.input, out)
    for layer in m.layers: layer.trainable=False
    return m

def get_inception():
    base = InceptionV3(weights='imagenet', include_top=False, input_shape=(IMG_SIZE[0],IMG_SIZE[1],3))
    out = GlobalAveragePooling2D()(base.output); m=Model(base.input, out)
    for layer in m.layers: layer.trainable=False
    return m

dn_model = get_densenet(); res_model = get_resnet(); inc_model = get_inception()
print("Backbones loaded: densenet,res,inc")


Backbones loaded: densenet,res,inc


In [6]:
def extract_dri_features(items, cache_name):
    os.makedirs("DRI_cache", exist_ok=True)
    Xf = f"DRI_cache/{cache_name}_X.npy"; yf = f"DRI_cache/{cache_name}_y.npy"
    if os.path.exists(Xf) and os.path.exists(yf):
        X = np.load(Xf); y = np.load(yf, allow_pickle=True); return X,y
    feats=[]; labels=[]
    for path,label in tqdm(items, desc=f"Extract {cache_name}"):
        arr = load_img_array(path)
        a1 = preprocess_for_backbone(arr, 'densenet'); f1 = dn_model.predict(np.expand_dims(a1,0), verbose=0)[0]
        a2 = preprocess_for_backbone(arr, 'resnet'); f2 = res_model.predict(np.expand_dims(a2,0), verbose=0)[0]
        a3 = preprocess_for_backbone(arr, 'inception'); f3 = inc_model.predict(np.expand_dims(a3,0), verbose=0)[0]
        feat = np.concatenate([f1,f2,f3]); feats.append(feat); labels.append(label)
    X=np.vstack(feats); y=np.array(labels); np.save(Xf,X); np.save(yf,y); return X,y

X_train_dri, y_train_dri = extract_dri_features(train_items, "train")
X_val_dri, y_val_dri = extract_dri_features(val_items, "val")
X_test_dri, y_test_dri = extract_dri_features(test_items, "test")
print("Feature shapes:", X_train_dri.shape, X_val_dri.shape, X_test_dri.shape)


Extract test: 100%|███████████████████████████████████████████████████████████████████████████████| 662/662 [07:09<00:00,  1.54it/s]

Feature shapes: (3322, 6016) (444, 6016) (662, 6016)


In [7]:
le = LabelEncoder(); y_train_enc = le.fit_transform(y_train_dri); y_val_enc = le.transform(y_val_dri); y_test_enc = le.transform(y_test_dri)

from tensorflow.keras import Input
inp = Input(shape=(X_train_dri.shape[1],))
x = Dense(512, activation='relu')(inp); x = Dropout(0.5)(x)
out = Dense(len(CLASSES), activation='softmax')(x)
head1 = Model(inp,out)
head1.compile(optimizer=tf.keras.optimizers.Adam(LR), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
head1.fit(X_train_dri, y_train_enc, validation_data=(X_val_dri,y_val_enc), epochs=EPOCHS_STAGE1, batch_size=BATCH_SIZE)




Epoch 1/10
208/208 [==============================] - 1s 4ms/step - loss: 0.7210 - accuracy: 0.8390 - val_loss: 0.4434 - val_accuracy: 0.8896
Epoch 2/10
208/208 [==============================] - 1s 3ms/step - loss: 0.2140 - accuracy: 0.9380 - val_loss: 0.2714 - val_accuracy: 0.9167
Epoch 3/10
208/208 [==============================] - 1s 3ms/step - loss: 0.1431 - accuracy: 0.9542 - val_loss: 0.3218 - val_accuracy: 0.9144
Epoch 4/10
208/208 [==============================] - 1s 3ms/step - loss: 0.1002 - accuracy: 0.9684 - val_loss: 0.2775 - val_accuracy: 0.9257
Epoch 5/10
208/208 [==============================] - 1s 3ms/step - loss: 0.0800 - accuracy: 0.9756 - val_loss: 0.2687 - val_accuracy: 0.9324
Epoch 6/10
208/208 [==============================] - 1s 3ms/step - loss: 0.0579 - accuracy: 0.9795 - val_loss: 0.3715 - val_accuracy: 0.9099
Epoch 7/10
208/208 [==============================] - 1s 3ms/step - loss: 0.0523 - accuracy: 0.9831 - val_loss: 0.2735 - val_accuracy: 0.9122
Epoch 

In [8]:
tr_acc = head1.evaluate(X_train_dri, y_train_enc, verbose=0)[1]
val_acc = head1.evaluate(X_val_dri, y_val_enc, verbose=0)[1]
test_loss, test_acc = head1.evaluate(X_test_dri, y_test_enc, verbose=0)
y_test_pred = np.argmax(head1.predict(X_test_dri), axis=1)
pr = precision_score(y_test_enc, y_test_pred, average='macro', zero_division=0)
rc = recall_score(y_test_enc, y_test_pred, average='macro', zero_division=0)
f1 = f1_score(y_test_enc, y_test_pred, average='macro', zero_division=0)

result1 = pd.DataFrame([{'ensemble':'DRI-2025','train_acc':round(tr_acc*100,2),'val_acc':round(val_acc*100,2),'test_acc':round(test_acc*100,2),'precision':round(pr*100,2),'recall':round(rc*100,2),'f1':round(f1*100,2)}])
result1.to_csv("DRI_outputs/Table8_DRI_stage1.csv", index=False)
display(result1)

21/21 [==============================] - 0s 1ms/step


,ensemble,train_acc,val_acc,test_acc,precision,recall,f1
0,DRI-2025,99.94,92.79,89.88,91.36,90.98,90.57


In [9]:
sm = SMOTE(random_state=RANDOM_STATE, n_jobs=-1)
X_train_sm, y_train_sm = sm.fit_resample(X_train_dri, y_train_enc)

dtrain = xgb.DMatrix(X_train_sm, label=y_train_sm)
params = {'objective':'multi:softprob','num_class':len(CLASSES),'eta':0.1,'max_depth':6,'verbosity':0,'eval_metric':'mlogloss'}
bst = xgb.train(params, dtrain, num_boost_round=200)
joblib.dump(bst, "DRI_outputs/xgb_DRI.bst")


['DRI_outputs/xgb_DRI.bst']

In [10]:
imp = bst.get_score(importance_type='gain')
feat_imp = np.zeros(X_train_dri.shape[1])
for k,v in imp.items():
    idx = int(k.replace('f','')); feat_imp[idx]=v
K = min(TOPK, X_train_dri.shape[1]); topk_idx = np.argsort(feat_imp)[-K:][::-1]
np.save("DRI_outputs/topk_idx_DRI.npy", topk_idx)


In [11]:
X_train_sel = X_train_sm[:, topk_idx]; X_val_sel = X_val_dri[:, topk_idx]; X_test_sel = X_test_dri[:, topk_idx]
scaler_stage2 = StandardScaler(); X_train_sel_s = scaler_stage2.fit_transform(X_train_sel); X_val_sel_s = scaler_stage2.transform(X_val_sel); X_test_sel_s = scaler_stage2.transform(X_test_sel)
joblib.dump(scaler_stage2, "DRI_outputs/scaler_stage2.pkl")


['DRI_outputs/scaler_stage2.pkl']

In [12]:
from tensorflow.keras import Input
inp2 = Input(shape=(X_train_sel_s.shape[1],))
y = Dense(256, activation='relu')(inp2); y = Dropout(0.4)(y)
out2 = Dense(len(CLASSES), activation='softmax')(y)
head2 = Model(inp2,out2)
head2.compile(optimizer=tf.keras.optimizers.Adam(LR), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
head2.fit(X_train_sel_s, y_train_sm, validation_data=(X_val_sel_s, y_val_enc), epochs=EPOCHS_STAGE2, batch_size=BATCH_SIZE)


Epoch 1/10
254/254 [==============================] - 1s 4ms/step - loss: 0.5374 - accuracy: 0.8070 - val_loss: 0.3372 - val_accuracy: 0.8896
Epoch 2/10
254/254 [==============================] - 1s 3ms/step - loss: 0.1775 - accuracy: 0.9435 - val_loss: 0.2901 - val_accuracy: 0.9099
Epoch 3/10
254/254 [==============================] - 1s 3ms/step - loss: 0.1365 - accuracy: 0.9571 - val_loss: 0.2603 - val_accuracy: 0.9257
Epoch 4/10
254/254 [==============================] - 1s 3ms/step - loss: 0.1064 - accuracy: 0.9696 - val_loss: 0.2483 - val_accuracy: 0.9392
Epoch 5/10
254/254 [==============================] - 1s 3ms/step - loss: 0.0918 - accuracy: 0.9743 - val_loss: 0.2369 - val_accuracy: 0.9369
Epoch 6/10
254/254 [==============================] - 1s 3ms/step - loss: 0.0758 - accuracy: 0.9768 - val_loss: 0.2328 - val_accuracy: 0.9347
Epoch 7/10
254/254 [==============================] - 1s 3ms/step - loss: 0.0669 - accuracy: 0.9788 - val_loss: 0.2359 - val_accuracy: 0.9324
Epoch 

In [13]:
tr_acc2 = head2.evaluate(X_train_sel_s, y_train_sm, verbose=0)[1]
val_acc2 = head2.evaluate(X_val_sel_s, y_val_enc, verbose=0)[1]
test_loss2, test_acc2 = head2.evaluate(X_test_sel_s, y_test_enc, verbose=0)
y_test_pred2 = np.argmax(head2.predict(X_test_sel_s), axis=1)
pr2 = precision_score(y_test_enc, y_test_pred2, average='macro', zero_division=0)
rc2 = recall_score(y_test_enc, y_test_pred2, average='macro', zero_division=0)
f12 = f1_score(y_test_enc, y_test_pred2, average='macro', zero_division=0)

result2 = pd.DataFrame([{'ensemble':'DRI-2025','train_acc':round(tr_acc2*100,2),'val_acc':round(val_acc2*100,2),'test_acc':round(test_acc2*100,2),'precision':round(pr2*100,2),'recall':round(rc2*100,2),'f1':round(f12*100,2),'selected_features':int(K)}])
result2.to_csv("DRI_outputs/Table10_DRI_stage2.csv", index=False)
display(result2)


21/21 [==============================] - 0s 1ms/step


,ensemble,train_acc,val_acc,test_acc,precision,recall,f1,selected_features
0,DRI-2025,99.38,93.24,92.6,93.13,93.28,92.98,300


In [14]:
scaler_svm = StandardScaler()
X_train_svm = scaler_svm.fit_transform(X_train_sel)
X_test_svm = scaler_svm.transform(X_test_sel)
joblib.dump(scaler_svm, "DRI_outputs/scaler_svm.pkl")


['DRI_outputs/scaler_svm.pkl']

In [15]:
svm = SVC(kernel='rbf', C=10.0, gamma='scale', probability=True, random_state=RANDOM_STATE)
svm.fit(X_train_svm, y_train_sm)
joblib.dump(svm, "DRI_outputs/svm_DRI.pkl")


['DRI_outputs/svm_DRI.pkl']

In [16]:
y_train_pred_svm = svm.predict(X_train_svm)
y_test_pred_svm = svm.predict(X_test_svm)
tr_acc_svm = accuracy_score(y_train_sm, y_train_pred_svm)
test_acc_svm = accuracy_score(y_test_enc, y_test_pred_svm)
pr_svm = precision_score(y_test_enc, y_test_pred_svm, average='macro', zero_division=0)
rc_svm = recall_score(y_test_enc, y_test_pred_svm, average='macro', zero_division=0)
f1_svm = f1_score(y_test_enc, y_test_pred_svm, average='macro', zero_division=0)


In [17]:
result3 = pd.DataFrame([{'ensemble':'DRI-2025','train_acc':round(tr_acc_svm*100,2),'test_acc':round(test_acc_svm*100,2),'precision':round(pr_svm*100,2),'recall':round(rc_svm*100,2),'f1':round(f1_svm*100,2)}])
result3.to_csv("DRI_outputs/Table11_DRI_stage3.csv", index=False)
display(result3)


,ensemble,train_acc,test_acc,precision,recall,f1
0,DRI-2025,100.0,93.05,93.56,93.59,93.39


In [18]:
report = classification_report(y_test_enc, y_test_pred_svm, target_names=le.classes_, output_dict=True, zero_division=0)
rows=[]
for cls in le.classes_:
    d = report[cls]; rows.append({'class':cls, 'precision':round(d['precision']*100,2), 'recall':round(d['recall']*100,2), 'f1-score':round(d['f1-score']*100,2), 'support':int(d['support'])})
result4 = pd.DataFrame(rows); result4.to_csv("DRI_outputs/Table13_class_report.csv", index=False)
display(result4)


,class,precision,recall,f1-score,support
0,cyclone,98.54,97.12,97.83,139
1,earthquake,96.13,86.57,91.10,201
2,flood,82.70,95.62,88.70,160
3,wildfire,96.86,95.06,95.95,162
